# Scheduled report (example job)

A stand-in for real scheduled analysis, run headless by `run-compose-job.bat` with `jupyter nbconvert --execute`.

**Data:** synthetic. A day of simulated sensor readings, seeded from the run date, so each day's run differs but can be reproduced.
**Outputs:** a CSV of hourly statistics and a PNG chart in `/home/jovyan/output`, named after the run timestamp.
**Parameters** come from environment variables: `RUN_STAMP` (set by the launcher) and `FAIL=1` (make the run fail, to test error handling).


In [ ]:
import os
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")                      # headless: no display in a scheduled run
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_STAMP = os.environ.get("RUN_STAMP", datetime.now().strftime("%Y%m%d-%H%M%S"))
OUTPUT = Path("/home/jovyan/output")
OUTPUT.mkdir(parents=True, exist_ok=True)

if os.environ.get("FAIL") == "1":
    raise RuntimeError("FAIL=1 set: simulating a failed run")
print(f"run {RUN_STAMP}")


In [ ]:
# One reading per minute for 24 h: a daily cycle, slow drift and noise.
rng = np.random.default_rng(int(RUN_STAMP[:8]))          # seed = run date
minutes = pd.date_range(RUN_STAMP[:8], periods=24 * 60, freq="min")
t = np.arange(minutes.size) / 60.0                        # hours
value = 20 + 3 * np.sin(2 * np.pi * (t - 9) / 24) + 0.02 * t + rng.normal(0, 0.4, t.size)
readings = pd.Series(value, index=minutes, name="value")

hourly = readings.resample("h").agg(["mean", "min", "max", "std"]).round(3)
hourly.to_csv(OUTPUT / f"report_{RUN_STAMP}.csv")
hourly.head()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(readings.index, readings, lw=0.5, alpha=0.5, label="readings")
ax.plot(hourly.index, hourly["mean"], lw=2, label="hourly mean")
ax.set_title(f"Synthetic readings, run {RUN_STAMP}")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / f"report_{RUN_STAMP}.png", dpi=100)
print(f"daily mean {readings.mean():.2f}, min {readings.min():.2f}, max {readings.max():.2f}")
print("wrote", sorted(p.name for p in OUTPUT.glob(f"report_{RUN_STAMP}.*")))
